# Predicting Vaccination Rates Using Behavioral Data

This notebook implements a comprehensive pipeline for multilabel probability prediction of vaccine uptake from NHFS survey responses.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from config import CONFIG, ensure_output_dirs
from data_utils import load_joined_data, standardize_missing, add_strata_key, make_folds, preprocess_for_baseline, preprocess_for_bayesian, get_X_y
from model_utils import fit_logistic, fit_random_forest, fit_bayesian
from eval_utils import summarize_metrics, plot_roc_curves, plot_calibration, write_outputs
print('✓ Modules loaded')

✓ Modules loaded


In [2]:
ensure_output_dirs()
df = load_joined_data()
df = standardize_missing(df)
df = add_strata_key(df, CONFIG['outcomes'])
print(f'Loaded {len(df)} respondents')
print(f'h1n1 rate: {df["h1n1_vaccine"].mean():.3f}, seasonal rate: {df["seasonal_vaccine"].mean():.3f}')

Loaded 26707 respondents
h1n1 rate: 0.212, seasonal rate: 0.466


In [3]:
# Quick test mode: reduce runtime for iterative debugging.
QUICK_TEST_MODE = False

if QUICK_TEST_MODE:
    # Ensure stratification key exists even if cells were run out of order.
    if '_strata_key' not in df.columns:
        df = add_strata_key(df, CONFIG['outcomes'])

    # Downsample while preserving joint-label balance as much as possible.
    target_n = min(len(df), 4000)
    base_n = len(df)
    df = (
        df.groupby('_strata_key', group_keys=False)
          .apply(lambda g: g.sample(max(1, int(round(target_n * len(g) / base_n))), random_state=CONFIG['seed']))
          .reset_index(drop=True)
    )

    # Keep CV/model settings lightweight for quick checks.
    CONFIG['folds'] = 2
    CONFIG['max_bayes_folds'] = 0

    print('Quick test mode enabled')
    print(f"- rows after sampling: {len(df)}")
    print(f"- folds: {CONFIG['folds']}")
    print(f"- bayesian folds: {CONFIG['max_bayes_folds']}")
else:
    print('Quick test mode disabled: using full dataset and full settings')

Quick test mode disabled: using full dataset and full settings


In [4]:
if '_strata_key' not in df.columns:
    df = add_strata_key(df, CONFIG['outcomes'])

folds = make_folds(df, n_folds=CONFIG['folds'], seed=CONFIG['seed'])
print(f'Created {len(folds)} stratified folds')

Created 3 stratified folds


In [5]:
all_predictions = []
bayes_diagnostics = []
bayes_effect_summaries = []

for fold_idx, fold_data in enumerate(folds):
    print(f'Fold {fold_idx+1}/{len(folds)}')
    train_idx, test_idx = fold_data['train'], fold_data['test']
    train_df, test_df = df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)

    for outcome in CONFIG['outcomes']:
        train_base, test_base = preprocess_for_baseline(train_df, test_df, outcome)
        X_train_base, y_train_base = get_X_y(train_base, outcome)
        X_test_base, y_test_base = get_X_y(test_base, outcome)

        log_result = fit_logistic(X_train_base, y_train_base, X_test_base)
        log_preds = np.clip(log_result['predictions'], 1e-6, 1-1e-6)
        all_predictions.append(pd.DataFrame({
            'respondent_id': test_df['respondent_id'].values,
            'fold': fold_idx+1,
            'outcome': outcome,
            'model': 'logistic_ridge',
            'truth': y_test_base.values,
            'prediction': log_preds,
        }))

        rf_result = fit_random_forest(X_train_base, y_train_base, X_test_base, seed=CONFIG['seed']+fold_idx)
        rf_preds = np.clip(rf_result['predictions'], 1e-6, 1-1e-6)
        all_predictions.append(pd.DataFrame({
            'respondent_id': test_df['respondent_id'].values,
            'fold': fold_idx+1,
            'outcome': outcome,
            'model': 'random_forest',
            'truth': y_test_base.values,
            'prediction': rf_preds,
        }))

        # Bayesian hierarchical model and diagnostics (analysis plan).
        if fold_idx < CONFIG['max_bayes_folds']:
            train_bayes, test_bayes = preprocess_for_bayesian(train_df, test_df, outcome)
            bayes_result = fit_bayesian(train_bayes, test_bayes, outcome)
            bayes_preds = np.clip(bayes_result['predictions'], 1e-6, 1-1e-6)

            all_predictions.append(pd.DataFrame({
                'respondent_id': test_bayes['respondent_id'].values,
                'fold': fold_idx+1,
                'outcome': outcome,
                'model': 'bayesian_hierarchical',
                'truth': test_bayes[outcome].values,
                'prediction': bayes_preds,
                'pred_ci_lower': bayes_result['prediction_ci_lower'],
                'pred_ci_upper': bayes_result['prediction_ci_upper'],
            }))

            bayes_diagnostics.append({
                'fold': fold_idx + 1,
                'outcome': outcome,
                **bayes_result['diagnostics'],
            })

            sigma_summary = bayes_result['sigma_u_summary']
            if sigma_summary is not None:
                bayes_effect_summaries.append({
                    'fold': fold_idx + 1,
                    'outcome': outcome,
                    'group_sd_mean': float(np.asarray(sigma_summary['mean']).squeeze()),
                    'group_sd_ci_lower': float(np.asarray(sigma_summary['ci_lower']).squeeze()),
                    'group_sd_ci_upper': float(np.asarray(sigma_summary['ci_upper']).squeeze()),
                })

            beta_summary = bayes_result.get('beta_summary')
            if beta_summary is not None:
                beta_means = np.asarray(beta_summary['mean']).ravel()
                feature_names = bayes_result.get('feature_names', [])
                if len(feature_names) == len(beta_means) and len(feature_names) > 0:
                    top_idx = np.argsort(np.abs(beta_means))[-3:][::-1]
                    for j in top_idx:
                        bayes_effect_summaries.append({
                            'fold': fold_idx + 1,
                            'outcome': outcome,
                            'feature': feature_names[j],
                            'beta_mean': float(beta_means[j]),
                            'beta_ci_lower': float(np.asarray(beta_summary['ci_lower']).ravel()[j]),
                            'beta_ci_upper': float(np.asarray(beta_summary['ci_upper']).ravel()[j]),
                        })

pred_df = pd.concat(all_predictions, ignore_index=True)
print(f'✓ Training complete, collected {len(pred_df)} predictions')
print(f'✓ Bayesian fits completed: {len(bayes_diagnostics)}')
print(f"✓ Bayesian prior scale in use (for sensitivity analysis): {CONFIG['bayes'].get('prior_sd', 1.0)}")

Fold 1/3
Building...



Building: found in cache, done.Messages from stanc:
    provided, or the prior(s) depend on data variables. In the later case,
    this may be a false positive.
Sampling:   0%
Sampling:   0% (1/2100)
Sampling:   0% (2/2100)
Sampling:   5% (101/2100)
Sampling:  10% (200/2100)
Sampling:  14% (300/2100)
Sampling:  19% (400/2100)
Sampling:  24% (500/2100)
Sampling:  29% (600/2100)
Sampling:  31% (651/2100)
Sampling:  33% (702/2100)
Sampling:  38% (801/2100)
Sampling:  43% (900/2100)
Sampling:  48% (1000/2100)
Sampling:  52% (1100/2100)
Sampling:  57% (1200/2100)
Sampling:  62% (1300/2100)
Sampling:  67% (1400/2100)
Sampling:  71% (1500/2100)
Sampling:  76% (1600/2100)
Sampling:  81% (1700/2100)
Sampling:  86% (1800/2100)
Sampling:  90% (1900/2100)
Sampling:  95% (2000/2100)
Sampling: 100% (2100/2100)
Sampling: 100% (2100/2100), done.
Messages received during sampling:
  Gradient evaluation took 0.000984 seconds
  1000 transitions using 10 leapfrog steps per transition would take 9.84 seco

Building...



Building: found in cache, done.Messages from stanc:
    provided, or the prior(s) depend on data variables. In the later case,
    this may be a false positive.
Sampling:   0%
Sampling:   0% (1/2100)
Sampling:   0% (2/2100)
Sampling:   5% (101/2100)
Sampling:  10% (200/2100)
Sampling:  14% (300/2100)
Sampling:  19% (400/2100)
Sampling:  24% (500/2100)
Sampling:  29% (600/2100)
Sampling:  31% (651/2100)
Sampling:  33% (702/2100)
Sampling:  38% (801/2100)
Sampling:  43% (900/2100)
Sampling:  48% (1000/2100)
Sampling:  52% (1100/2100)
Sampling:  57% (1200/2100)
Sampling:  62% (1300/2100)
Sampling:  67% (1400/2100)
Sampling:  71% (1500/2100)
Sampling:  76% (1600/2100)
Sampling:  81% (1700/2100)
Sampling:  86% (1800/2100)
Sampling:  90% (1900/2100)
Sampling:  95% (2000/2100)
Sampling: 100% (2100/2100)
Sampling: 100% (2100/2100), done.
Messages received during sampling:
  Gradient evaluation took 0.001016 seconds
  1000 transitions using 10 leapfrog steps per transition would take 10.16 sec

Fold 2/3
Fold 3/3
✓ Training complete, collected 124634 predictions
✓ Bayesian fits completed: 2
✓ Bayesian prior scale in use (for sensitivity analysis): 1.0


In [6]:
metrics = summarize_metrics(pred_df)
print(metrics.to_string())
write_outputs(pred_df, metrics)
print('✓ Metrics saved')

if bayes_diagnostics:
    bayes_diag_df = pd.DataFrame(bayes_diagnostics)
    print('\nBayesian convergence diagnostics (target: R_hat near 1.00, strong ESS):')
    print(bayes_diag_df.to_string(index=False))

if bayes_effect_summaries:
    bayes_effect_df = pd.DataFrame(bayes_effect_summaries)
    print('\nGroup-level random effect uncertainty (sigma_u):')
    print(bayes_effect_df.to_string(index=False))

if {'pred_ci_lower', 'pred_ci_upper'}.issubset(pred_df.columns):
    bayes_pred = pred_df[pred_df['model'] == 'bayesian_hierarchical'].copy()
    if not bayes_pred.empty:
        mean_ci_width = (bayes_pred['pred_ci_upper'] - bayes_pred['pred_ci_lower']).mean()
        print(f"\nBayesian posterior predictive check: mean 95% CI width = {mean_ci_width:.4f}")

                   model           outcome       auc     brier        n  macro_auc  macro_brier
0  bayesian_hierarchical      h1n1_vaccine  0.833249  0.118609   8903.0   0.839329     0.138811
1  bayesian_hierarchical  seasonal_vaccine  0.845408  0.159013   8903.0   0.839329     0.138811
2         logistic_ridge      h1n1_vaccine  0.871029  0.107442  26707.0   0.873418     0.124415
3         logistic_ridge  seasonal_vaccine  0.875807  0.141389  26707.0   0.873418     0.124415
4          random_forest      h1n1_vaccine  0.870228  0.111647  26707.0   0.872584     0.129692
5          random_forest  seasonal_vaccine  0.874939  0.147737  26707.0   0.872584     0.129692
✓ Metrics saved

Bayesian convergence diagnostics (target: R_hat near 1.00, strong ESS):
 fold          outcome  max_rhat  min_bulk_ess  min_tail_ess
    1     h1n1_vaccine       NaN           NaN           NaN
    1 seasonal_vaccine       NaN           NaN           NaN

Group-level random effect uncertainty (sigma_u):
 fold 

In [7]:
plot_roc_curves(pred_df, CONFIG['paths']['roc_plot'])
plot_calibration(pred_df, CONFIG['paths']['calibration_plot'])
print('✓ Plots generated')

✓ Plots generated


## Bayesian Diagnostics Snapshot

This section aligns with the analysis plan by checking convergence and uncertainty outputs from MCMC.

- Target convergence: max R-hat close to 1.00 and sufficiently large ESS.
- Posterior predictive uncertainty: 95% credible interval width for predicted probabilities.
- Group-level uncertainty: posterior interval for the random-effect scale parameter (sigma_u).

In [10]:
from pathlib import Path

R_HAT_THRESHOLD = 1.01
ESS_THRESHOLD = 400

if not bayes_diagnostics:
    print('No Bayesian diagnostics available. Increase CONFIG["max_bayes_folds"] and rerun training.')
else:
    bayes_diag_df = pd.DataFrame(bayes_diagnostics).copy()

    def resolve_metric_col(df, aliases):
        for name in aliases:
            if name in df.columns:
                return name
        return None

    metric_aliases = {
        'rhat': ['rhat_max', 'max_rhat'],
        'ess_bulk': ['ess_bulk_min', 'min_bulk_ess'],
        'ess_tail': ['ess_tail_min', 'min_tail_ess'],
        'divergences': ['divergences', 'n_divergent'],
        'treedepth': ['treedepth_saturated', 'n_max_treedepth'],
    }

    resolved = {k: resolve_metric_col(bayes_diag_df, v) for k, v in metric_aliases.items()}

    for key, col in resolved.items():
        if col is not None:
            bayes_diag_df[col] = pd.to_numeric(bayes_diag_df[col], errors='coerce')

    rhat_series = bayes_diag_df[resolved['rhat']] if resolved['rhat'] else pd.Series(np.nan, index=bayes_diag_df.index)
    ess_bulk_series = bayes_diag_df[resolved['ess_bulk']] if resolved['ess_bulk'] else pd.Series(np.nan, index=bayes_diag_df.index)
    ess_tail_series = bayes_diag_df[resolved['ess_tail']] if resolved['ess_tail'] else pd.Series(np.nan, index=bayes_diag_df.index)
    divergences_series = bayes_diag_df[resolved['divergences']] if resolved['divergences'] else pd.Series(0, index=bayes_diag_df.index)
    treedepth_series = bayes_diag_df[resolved['treedepth']] if resolved['treedepth'] else pd.Series(0, index=bayes_diag_df.index)

    bayes_diag_df['rhat_ok'] = rhat_series <= R_HAT_THRESHOLD
    bayes_diag_df['ess_bulk_ok'] = ess_bulk_series >= ESS_THRESHOLD
    bayes_diag_df['ess_tail_ok'] = ess_tail_series >= ESS_THRESHOLD
    bayes_diag_df['hmc_ok'] = (divergences_series.fillna(0) == 0) & (treedepth_series.fillna(0) == 0)
    bayes_diag_df['all_checks_pass'] = (
        bayes_diag_df['rhat_ok']
        & bayes_diag_df['ess_bulk_ok']
        & bayes_diag_df['ess_tail_ok']
        & bayes_diag_df['hmc_ok']
    )

    bayes_diag_df['rhat_metric'] = rhat_series
    bayes_diag_df['ess_bulk_metric'] = ess_bulk_series
    bayes_diag_df['ess_tail_metric'] = ess_tail_series
    bayes_diag_df['divergences_metric'] = divergences_series
    bayes_diag_df['treedepth_metric'] = treedepth_series

    display_cols = [
        'fold', 'outcome', 'rhat_metric', 'ess_bulk_metric', 'ess_tail_metric',
        'divergences_metric', 'treedepth_metric', 'all_checks_pass'
    ]
    display_cols = [c for c in display_cols if c in bayes_diag_df.columns]

    print('Bayesian diagnostics per fit:')
    display(bayes_diag_df[display_cols].sort_values(['fold', 'outcome']).reset_index(drop=True))

    snapshot = {
        'n_fits': int(len(bayes_diag_df)),
        'all_checks_pass_rate': float(bayes_diag_df['all_checks_pass'].mean()),
        'max_rhat_observed': float(bayes_diag_df['rhat_metric'].max()),
        'min_ess_bulk_observed': float(bayes_diag_df['ess_bulk_metric'].min()),
        'min_ess_tail_observed': float(bayes_diag_df['ess_tail_metric'].min()),
        'total_divergences': int(bayes_diag_df['divergences_metric'].fillna(0).sum()),
        'total_treedepth_saturated': int(bayes_diag_df['treedepth_metric'].fillna(0).sum()),
    }

    if {'pred_ci_lower', 'pred_ci_upper'}.issubset(pred_df.columns):
        bayes_pred = pred_df[pred_df['model'] == 'bayesian_hierarchical'].copy()
        if not bayes_pred.empty:
            ci_width = bayes_pred['pred_ci_upper'] - bayes_pred['pred_ci_lower']
            snapshot['mean_pred_ci_width'] = float(ci_width.mean())
            snapshot['median_pred_ci_width'] = float(ci_width.median())

    if bayes_effect_summaries:
        sigma_df = pd.DataFrame(bayes_effect_summaries)
        sigma_df = sigma_df.dropna(subset=['group_sd_mean']) if 'group_sd_mean' in sigma_df.columns else sigma_df
        if not sigma_df.empty and 'group_sd_mean' in sigma_df.columns:
            snapshot['mean_group_sd'] = float(pd.to_numeric(sigma_df['group_sd_mean'], errors='coerce').mean())

    snapshot_df = pd.DataFrame([snapshot])
    print('\nBayesian diagnostics snapshot:')
    display(snapshot_df)

    out_dir = Path(CONFIG['paths']['metrics']).parent
    out_dir.mkdir(parents=True, exist_ok=True)
    snapshot_path = out_dir / 'bayesian_diagnostics_snapshot.csv'
    details_path = out_dir / 'bayesian_diagnostics_per_fit.csv'
    snapshot_df.to_csv(snapshot_path, index=False)
    bayes_diag_df.to_csv(details_path, index=False)
    print(f'Saved snapshot: {snapshot_path}')
    print(f'Saved per-fit diagnostics: {details_path}')

Bayesian diagnostics per fit:


,fold,outcome,rhat_metric,ess_bulk_metric,ess_tail_metric,divergences_metric,treedepth_metric,all_checks_pass
0,1,h1n1_vaccine,NaN,NaN,NaN,0,0,False
1,1,seasonal_vaccine,NaN,NaN,NaN,0,0,False



Bayesian diagnostics snapshot:


,n_fits,all_checks_pass_rate,max_rhat_observed,min_ess_bulk_observed,min_ess_tail_observed,total_divergences,total_treedepth_saturated,mean_pred_ci_width,median_pred_ci_width,mean_group_sd
0,2,0.0,NaN,NaN,NaN,0,0,0.054678,0.052489,0.055261


Saved snapshot: /Users/silasfaldenew/GitHub/datasci-451-final-project/outputs/metrics/bayesian_diagnostics_snapshot.csv
Saved per-fit diagnostics: /Users/silasfaldenew/GitHub/datasci-451-final-project/outputs/metrics/bayesian_diagnostics_per_fit.csv
